In [1]:
import pandas as pd
import os
import pickle
import bmra_prep
import bmra_prep.pathway_activity.prediction

In [11]:
cell_line ='BC3C_dec'

data_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}_aug/00_outputs_2020_{cell_line}/"
out_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}_aug/01_outputs_2020_{cell_line}/"


os.makedirs(out_dir, exist_ok = True)

# Load Data

In [12]:
out_dir

'/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_BC3C_dec_aug/01_outputs_2020_BC3C_dec/'

In [3]:
# load metdadata dict and extract used elements
with open(os.path.join(data_dir, "metadata.pickle"), "rb") as f:
    all_metadata = pickle.load(f)

n_modules = all_metadata["n_modules"]
n_genes = all_metadata["n_genes"]
n_experiments = all_metadata["n_experiments"]

modules = all_metadata["modules"]
exp_ids = all_metadata["exp_ids"]
genes = all_metadata["genes"]

In [4]:
# load data
L1000_df = pd.read_csv(
    os.path.join(data_dir, "L1000_Data_norm_data.csv"),
    index_col = 0,
)

x = L1000_df.values
x.shape

(978, 124)

In [5]:
# load doses and perturbation matrix
inhib_conc_matrix = pd.read_csv(
    os.path.join(data_dir, "inhib_conc_annotated.csv"),
    index_col = 0,
).values

ic50_matrix = pd.read_csv(
    os.path.join(data_dir, "ic50_annotated.csv"),
    index_col = 0,
).values

# gamma_matrix = pd.read_csv(
#     os.path.join(data_dir, "gamma_annotated.csv"),
#     index_col = 0,
# ).values

pert_matrix = pd.read_csv(
    os.path.join(data_dir, "pert_annotated.csv"),
    index_col = 0,
).values

In [6]:
# y_true = (1 + gamma_matrix * inhib_conc_matrix / ic50_matrix) / (1 + inhib_conc_matrix / ic50_matrix)

y_true = 1 / (1 + inhib_conc_matrix / ic50_matrix)

display(y_true.shape)
y_true

(11, 124)

array([[1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       ...,
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 0.41176471, 0.41176471,
        0.41176471]])

## Run models

In [7]:
a_coeffs = bmra_prep.pathway_activity.prediction.predict_coeffs(
    x, y_true, pert_matrix, 200_000, 10, 10, 10, 100)

In [8]:
a_coeffs_df = pd.DataFrame(a_coeffs, index = modules, columns = genes)
a_coeffs_df.to_csv(os.path.join(out_dir, "a_coeffs.csv"))
#a_coeffs_df = pd.read_csv(os.path.join(out_dir,'a_coeffs.csv'),index_col=0)
#a_coeffs = a_coeffs_df.values
display(a_coeffs_df.astype(bool).sum(axis='columns'))
display(a_coeffs_df)

Androgen      978
CDK1_2        978
CDK4_6        978
EGFR          978
Estrogen      978
FGFR          978
PI3K          978
p53           978
TOP2A         978
Src           978
TGFBR/SMAD    978
dtype: int64

,AARS,ABCB6,ABCC5,ABCF1,ABCF3,ABHD4,ABHD6,ABL1,ACAA1,ACAT2,...,ZMIZ1,ZMYM2,ZNF131,ZNF274,ZNF318,ZNF395,ZNF451,ZNF586,ZNF589,ZW10
Androgen,-0.000006,-1.318990e-05,-9.274144e-07,-0.000021,1.137053e-05,0.000012,3.361330e-05,0.000004,0.000002,3.227656e-05,...,-1.387871e-05,0.000035,0.000018,7.650875e-06,-1.015689e-05,0.000007,3.039423e-06,-1.077619e-05,-3.126226e-06,-0.000012
CDK1_2,0.000003,-8.132477e-06,4.912750e-06,0.000005,-8.672842e-06,0.000003,2.230308e-06,0.000017,-0.000003,1.008900e-07,...,3.018141e-06,-0.000008,-0.000005,-8.199845e-07,-6.935339e-06,0.000008,1.198113e-05,7.084565e-07,7.639642e-07,-0.000009
CDK4_6,0.000004,-1.801982e-05,-2.430049e-05,0.000009,8.344808e-07,0.000006,2.557970e-05,-0.000008,0.000004,1.472077e-05,...,-8.979369e-06,0.000005,0.000018,-1.784553e-05,8.461589e-06,0.000017,-8.623263e-06,3.326942e-05,1.543568e-05,-0.000006
EGFR,0.000001,1.852575e-05,-8.636222e-06,-0.000014,1.735729e-05,-0.000302,-1.573015e-05,0.000002,0.000025,-2.650484e-02,...,-3.410511e-05,-0.000005,-0.000002,-1.820187e-06,-9.500094e-06,0.000333,6.679933e-06,3.939613e-05,2.019638e-05,-0.000024
Estrogen,-0.000015,2.430977e-05,-4.367990e-06,-0.000021,2.585196e-06,-0.000001,1.924048e-07,0.000017,-0.000019,-2.732464e-01,...,5.573508e-06,0.000008,-0.000021,1.423370e-06,5.448300e-07,-0.000014,-6.926769e-06,-2.033139e-06,-5.226638e-06,0.000013
FGFR,-0.001219,-5.761032e-06,-2.198072e-05,-0.000012,-2.192454e-05,-0.000044,-5.816056e-06,-0.000018,0.000002,-1.476175e-05,...,-5.181211e-06,-0.000031,0.000014,-7.494357e-06,1.877134e-05,0.000014,1.250836e-05,1.302331e-05,9.571122e-06,0.000014
PI3K,0.000008,1.370047e-05,1.654374e-05,-0.000041,1.395012e-05,-0.000012,-5.924567e-06,0.000011,0.000004,-3.008158e-06,...,4.333137e-06,0.000013,0.000006,5.185849e-06,1.376313e-05,-0.000191,-6.459638e-06,-7.661554e-06,1.136065e-05,0.027251
p53,-0.000070,-4.896253e-06,-3.086745e-06,0.000012,-3.874474e-06,0.000016,2.255701e-01,0.000002,0.000004,9.192206e-06,...,1.316217e-05,-0.000012,-0.000051,2.209760e-05,1.034459e-05,-0.000011,7.476497e-06,-1.575867e-05,5.076953e-06,0.000008
TOP2A,-0.000012,1.680213e-05,-4.325030e-05,0.000016,1.738357e-05,0.000012,1.530485e-05,-0.000013,0.000020,-6.802811e-06,...,-4.989002e-07,0.000014,0.000004,1.287977e-05,6.087782e-05,-0.000004,-2.265042e-07,-1.250618e-05,9.276734e-06,0.000002
Src,0.000001,-1.390986e-05,2.435173e-05,0.000007,2.250115e-05,-0.000017,1.233997e-05,0.000004,0.000018,-1.642673e-05,...,1.017915e-05,0.000006,-0.000051,-9.444464e-06,-2.781788e-06,0.000017,-7.583642e-06,-8.075329e-06,3.708082e-05,-0.000007


In [9]:
#pathway_activity = a_coeffs @ x
#pathway_activity.shape

In [9]:
R_global = bmra_prep.pathway_activity.calc_global_response_from_pathway_activity(
    bmra_prep.pathway_activity.calc_pathway_activity(x,a_coeffs),
    modules,
    L1000_df.columns
)
R_global_df = R_global.dataframe
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,TGFBR/SMAD_V11,TGFBR/SMAD_V12,TGFBR/SMAD_V13,TGFBR/SMAD_V14,TGFBR/SMAD_V15,TGFBR/SMAD_V16,TGFBR/SMAD_V17,TGFBR/SMAD_V18,TGFBR/SMAD_V19,TGFBR/SMAD_V20
Androgen,-0.052671,-0.009376,0.001865,-0.039650,0.001388,0.020341,0.004221,0.016370,0.020991,0.036097,...,0.013393,0.005400,-0.001662,0.001754,0.000742,0.008326,-0.003464,0.009006,0.003900,0.022117
CDK1_2,-0.903266,-0.682124,0.060050,-0.122889,0.112326,-0.308019,-0.005712,-0.242859,0.119589,0.041190,...,-0.442907,-0.407670,-0.373295,-0.390382,-0.361692,-0.376317,-0.368947,-0.406712,-0.404733,-0.438361
CDK4_6,-0.092655,-0.211609,-0.265543,-0.141419,-0.163855,-0.102997,-0.017022,-0.028904,-0.218487,-0.010460,...,0.197639,0.188714,0.177176,0.140548,0.138619,0.150254,0.171738,0.164223,0.189630,0.162679
EGFR,0.597922,0.498119,0.229146,0.371282,0.486497,0.113911,-0.415709,0.263542,0.286880,-0.046658,...,-1.223360,-0.819891,-0.805609,-0.705743,-0.676213,-0.691545,-0.743427,-0.897886,-0.877888,-0.931579
Estrogen,-0.123941,-0.207504,-0.210261,-0.409124,-0.946507,-0.310670,-0.085796,-0.237827,-0.163861,-0.040535,...,-0.296308,-0.265374,-0.238298,-0.262718,-0.242992,-0.251438,-0.229116,-0.275995,-0.259764,-0.304067
FGFR,-0.111760,-0.177340,-0.089483,0.051597,-0.027978,-0.407779,-0.033536,-0.027036,-0.061521,-0.300781,...,-0.859908,-0.704521,-0.642461,-0.642562,-0.572569,-0.621204,-0.628673,-0.715805,-0.721523,-0.797321
PI3K,-1.902204,-1.691679,-1.425107,-1.235516,-0.704825,0.290999,-0.146265,-0.195022,-0.845867,-0.279938,...,-0.178116,-0.139614,-0.119142,-0.114442,-0.098940,-0.139076,-0.129713,-0.136427,-0.155018,-0.182063
p53,-0.213495,-0.216330,-0.126155,-0.410001,0.044483,-1.628395,-1.475728,-0.120038,-0.090925,-1.328023,...,0.011914,-0.004709,0.008119,0.018305,-0.009165,0.011306,0.038291,-0.006784,0.033059,-0.003207
TOP2A,-0.227516,0.074485,-0.235513,-0.172464,-0.127025,0.054532,0.071828,-1.999776,-0.205288,-0.287793,...,0.120949,0.105629,0.097411,0.082543,0.076925,0.092263,0.098283,0.097782,0.109946,0.110146
Src,-0.929010,-1.678232,0.522948,-1.206300,0.571490,-1.128080,0.495607,0.394248,0.396986,0.447411,...,-0.128041,-0.154378,-0.098396,-0.128126,-0.056733,-0.118223,-0.099147,-0.114078,-0.156072,-0.166969


In [13]:
R_global_df.to_csv(os.path.join(out_dir, "R_global_annotated.csv"))
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,TGFBR/SMAD_V11,TGFBR/SMAD_V12,TGFBR/SMAD_V13,TGFBR/SMAD_V14,TGFBR/SMAD_V15,TGFBR/SMAD_V16,TGFBR/SMAD_V17,TGFBR/SMAD_V18,TGFBR/SMAD_V19,TGFBR/SMAD_V20
Androgen,-0.052671,-0.009376,0.001865,-0.039650,0.001388,0.020341,0.004221,0.016370,0.020991,0.036097,...,0.013393,0.005400,-0.001662,0.001754,0.000742,0.008326,-0.003464,0.009006,0.003900,0.022117
CDK1_2,-0.903266,-0.682124,0.060050,-0.122889,0.112326,-0.308019,-0.005712,-0.242859,0.119589,0.041190,...,-0.442907,-0.407670,-0.373295,-0.390382,-0.361692,-0.376317,-0.368947,-0.406712,-0.404733,-0.438361
CDK4_6,-0.092655,-0.211609,-0.265543,-0.141419,-0.163855,-0.102997,-0.017022,-0.028904,-0.218487,-0.010460,...,0.197639,0.188714,0.177176,0.140548,0.138619,0.150254,0.171738,0.164223,0.189630,0.162679
EGFR,0.597922,0.498119,0.229146,0.371282,0.486497,0.113911,-0.415709,0.263542,0.286880,-0.046658,...,-1.223360,-0.819891,-0.805609,-0.705743,-0.676213,-0.691545,-0.743427,-0.897886,-0.877888,-0.931579
Estrogen,-0.123941,-0.207504,-0.210261,-0.409124,-0.946507,-0.310670,-0.085796,-0.237827,-0.163861,-0.040535,...,-0.296308,-0.265374,-0.238298,-0.262718,-0.242992,-0.251438,-0.229116,-0.275995,-0.259764,-0.304067
FGFR,-0.111760,-0.177340,-0.089483,0.051597,-0.027978,-0.407779,-0.033536,-0.027036,-0.061521,-0.300781,...,-0.859908,-0.704521,-0.642461,-0.642562,-0.572569,-0.621204,-0.628673,-0.715805,-0.721523,-0.797321
PI3K,-1.902204,-1.691679,-1.425107,-1.235516,-0.704825,0.290999,-0.146265,-0.195022,-0.845867,-0.279938,...,-0.178116,-0.139614,-0.119142,-0.114442,-0.098940,-0.139076,-0.129713,-0.136427,-0.155018,-0.182063
p53,-0.213495,-0.216330,-0.126155,-0.410001,0.044483,-1.628395,-1.475728,-0.120038,-0.090925,-1.328023,...,0.011914,-0.004709,0.008119,0.018305,-0.009165,0.011306,0.038291,-0.006784,0.033059,-0.003207
TOP2A,-0.227516,0.074485,-0.235513,-0.172464,-0.127025,0.054532,0.071828,-1.999776,-0.205288,-0.287793,...,0.120949,0.105629,0.097411,0.082543,0.076925,0.092263,0.098283,0.097782,0.109946,0.110146
Src,-0.929010,-1.678232,0.522948,-1.206300,0.571490,-1.128080,0.495607,0.394248,0.396986,0.447411,...,-0.128041,-0.154378,-0.098396,-0.128126,-0.056733,-0.118223,-0.099147,-0.114078,-0.156072,-0.166969
